# Smart Tourist Routing and Risk Alert System
## Random Forest — next-day heavy-rainfall screening
**Research notebook.** This reproduces the VS Code training approach. It predicts whether next-day daily precipitation reaches **20 mm**, using preceding-day observed weather. **It does not predict floods, landslides, roadblocks, or official hazard warnings.** The 20 mm cutoff is an illustrative research definition, not an official warning threshold.

### 1. Install and import libraries

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay,
                             RocCurveDisplay, PrecisionRecallDisplay,
                             roc_auc_score, average_precision_score)

### 2. Upload the CSV
In Colab, run the cell and choose `historical_weather.csv` from your computer (`SmartTouristBackend/hazard_ml/`).

In [ ]:
from google.colab import files
uploaded = files.upload()
assert "historical_weather.csv" in uploaded, "Please upload historical_weather.csv"
df = pd.read_csv("historical_weather.csv", parse_dates=["date"])
print("Dataset shape:", df.shape)
display(df.head())

### 3. Inspect the dataset

In [ ]:
print("Columns:", list(df.columns))
print("Missing values: \n", df.isna().sum())
print("Locations: \n", df["location"].value_counts())
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Duplicate location/date rows:", df.duplicated(["location", "date"]).sum())

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df["precipitation_sum"].dropna(), bins=50)
plt.title("Distribution of daily precipitation")
plt.xlabel("Precipitation (mm)")
plt.ylabel("Number of days")
plt.tight_layout()
plt.show()

### 4. Create the next-day target and input features
Only information from the current day or earlier is used as model input. Tomorrow’s precipitation is used solely as the target. End-of-series records are removed.

In [ ]:
required = ["location","latitude","longitude","date","temperature_2m_max",
            "temperature_2m_min","precipitation_sum","rain_sum","wind_speed_10m_max"]
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f"Missing columns: {missing}")
if df.duplicated(["location","date"]).any(): raise ValueError("Duplicate location/date rows")
df = df.sort_values(["location","date"]).copy()
weather = ["temperature_2m_max","temperature_2m_min","precipitation_sum",
           "rain_sum","wind_speed_10m_max"]
for c in weather: df[c] = pd.to_numeric(df[c], errors="coerce")
g = df.groupby("location", sort=False)
df["next_day_precipitation_mm"] = g["precipitation_sum"].shift(-1)
df["next_date"] = g["date"].shift(-1)
df = df[(df["next_date"] - df["date"]).dt.days == 1].copy()
THRESHOLD_MM = 20.0
df["target_heavy_rain_tomorrow"] = (df["next_day_precipitation_mm"] >= THRESHOLD_MM).astype(int)
df["month_sin"] = np.sin(2*np.pi*df["date"].dt.month/12)
df["month_cos"] = np.cos(2*np.pi*df["date"].dt.month/12)
df["day_of_year_sin"] = np.sin(2*np.pi*df["date"].dt.dayofyear/365.25)
df["day_of_year_cos"] = np.cos(2*np.pi*df["date"].dt.dayofyear/365.25)
features = ["latitude","longitude",*weather,"month_sin","month_cos",
            "day_of_year_sin","day_of_year_cos"]
df = df.dropna(subset=features+["next_day_precipitation_mm"]).copy()
print("Model-ready rows:", len(df))
print("Heavy-rain class counts: \n", df["target_heavy_rain_tomorrow"].value_counts())

In [ ]:
counts = df["target_heavy_rain_tomorrow"].value_counts().reindex([0,1], fill_value=0)
plt.figure(figsize=(6,4))
plt.bar(["Below 20 mm", "20 mm or more"], counts.values)
plt.title("Next-day rainfall target distribution")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()

### 5. Split by time, not randomly
Train on 2023–2024, test on 2025. This helps prevent using future dates to train a model evaluated on earlier dates.

In [ ]:
train = df[df["date"] < "2025-01-01"].copy()
test = df[df["date"] >= "2025-01-01"].copy()
X_train, y_train = train[features], train["target_heavy_rain_tomorrow"]
X_test, y_test = test[features], test["target_heavy_rain_tomorrow"]
assert len(train)>0 and len(test)>0 and y_train.nunique()==2
print("Train:",len(train),"rows | Test:",len(test),"rows")
print("Heavy rain in train:",int(y_train.sum()),"| test:",int(y_test.sum()))

### 6. Train Random Forest and baseline
These hyperparameters match the VS Code script.

In [ ]:
model = RandomForestClassifier(n_estimators=250, max_depth=12, min_samples_leaf=5,
                               class_weight="balanced_subsample", random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
base_pred = baseline.predict(X_test)
print("Training complete")

### 7. Evaluate predictions

In [ ]:
def scores(actual, predicted):
    p,r,f,_ = precision_recall_fscore_support(actual,predicted,average="binary",zero_division=0)
    return {"accuracy":round(float(accuracy_score(actual,predicted)),4),
            "precision_heavy_rain":round(float(p),4),
            "recall_heavy_rain":round(float(r),4),
            "f1_heavy_rain":round(float(f),4),
            "confusion_matrix_0_1":confusion_matrix(actual,predicted,labels=[0,1]).tolist()}
metrics = {"task":"Next-day precipitation >= 20 mm using prior-day observed weather",
           "disclaimer":"Research screening only, not disaster or official warning prediction",
           "threshold_mm":THRESHOLD_MM,"train_period":"2023-2024","test_period":"2025",
           "train_rows":len(train),"test_rows":len(test),
           "train_positive_count":int(y_train.sum()),"test_positive_count":int(y_test.sum()),
           "random_forest":scores(y_test,pred),"majority_class_baseline":scores(y_test,base_pred)}
print("Random Forest:",metrics["random_forest"])
print("Baseline:",metrics["majority_class_baseline"])
print("\nClassification report:\n",classification_report(y_test,pred,zero_division=0))

### 8. Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,pred,labels=[0,1],
    display_labels=["Below 20 mm","20 mm or more"],values_format="d")
plt.title("Random Forest confusion matrix — 2025 test data")
plt.tight_layout()
plt.show()

### 9. ROC and precision–recall curves
Because heavy-rain events are uncommon, the precision–recall curve is especially informative.

In [ ]:
RocCurveDisplay.from_predictions(y_test,prob)
plt.title(f"ROC curve (AUC={roc_auc_score(y_test,prob):.3f})")
plt.tight_layout()
plt.show()

In [ ]:
PrecisionRecallDisplay.from_predictions(y_test,prob)
plt.title(f"Precision–recall curve (AP={average_precision_score(y_test,prob):.3f})")
plt.tight_layout()
plt.show()

### 10. Feature importance
Impurity-based feature importance is descriptive, not proof that a feature causes rainfall.

In [ ]:
importance = pd.Series(model.feature_importances_,index=features).sort_values()
plt.figure(figsize=(8,5))
plt.barh(importance.index,importance.values)
plt.title("Random Forest feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

### 11. Save model, metrics and predictions

In [ ]:
joblib.dump({"model":model,"features":features,"threshold_mm":THRESHOLD_MM,
             "task":metrics["task"]},"weather_risk_model.joblib")
with open("evaluation_metrics.json","w") as f: json.dump(metrics,f,indent=2)
test[["location","date","next_day_precipitation_mm","target_heavy_rain_tomorrow"]].assign(
    predicted_heavy_rain=pred,predicted_probability=prob).to_csv("test_predictions.csv",index=False)
print("Saved weather_risk_model.joblib, evaluation_metrics.json, test_predictions.csv")

### 12. Download outputs (optional)

In [ ]:
# Uncomment any line below to download its file
# files.download("weather_risk_model.joblib")
# files.download("evaluation_metrics.json")
# files.download("test_predictions.csv")

### Research interpretation
- The test set is from 2025, later than training years.
- High overall accuracy can be misleading when the positive class is rare.
- This experiment is a **next-day heavy-rainfall screening model**, not a validated flood/landslide predictor.
- A model using observed current-day weather is not a multi-day operational forecast without corresponding live inputs.
- For safety-critical alerts, incorporate official warnings and verified hazard reports; do not treat a negative model prediction as proof of safety.
- Before research submission, document the threshold choice, data source, locations, limitations, and comparisons with baselines.